## Estimating Homography with RANSAC

# Introduction to Homographies and Alignment

Welcome to the first unit of our course, **"Estimate Geometry Using Homographies and RANSAC with Python"**! Over the next four units, we are going to learn how to combine overlapping photographs to create a single, wide panorama.

To build a panorama, we need to mathematically align overlapping images. Imagine taking a map, tearing it into two pieces, and then trying to tape them back together. To align them correctly, you look for matching landmarks on both pieces.

In computer vision, the mathematical rule that aligns these two pieces is called a **homography**. A homography is a transformation matrix that maps points in one image to their corresponding points in another. By calculating this matrix, we can warp and shift our photos so they overlap perfectly.

> **Key Geometric Scenarios:**
> A homography is the correct model in two specific scenarios:
> 1. When the scene is **roughly planar** (like a flat wall).
> 2. When the camera undergoes **pure rotation around its optical center** (like panned shots on a tripod).
> 
> 
> If you move the camera's position in a scene with objects at many different depths, you encounter **parallax**, which can prevent a single homography from perfectly aligning the entire image.

---

## Handling Raw Matches

Before we can calculate our homography math, we need to find overlapping points. Imagine our code has already used helper functions (such as `detect_and_compute` and `match_descriptors`) to look at two photos, identify interesting spots like corners or edges, and pair them up.

However, raw matches are often imperfect. The computer might mistakenly match a cloud in the left image to a completely different cloud in the right image. These bad matches are called **outliers**. We cannot just trust all matches to build our map; we need a robust method to calculate our homography while rejecting bad data.

---

## Extracting Coordinates from Matches

To compute the alignment, we first extract the exact pixel $(x, y)$ coordinates of our matched points. The match objects provide indices used to look up the keypoint locations.

```python
def matched_points(kp1, kp2, matches):
    src_points = [kp1[m.queryIdx].pt for m in matches]
    dst_points = [kp2[m.trainIdx].pt for m in matches]

```

In this snippet:

* `kp1` and `kp2` are lists of **keypoints** (the detected feature locations) for the left and right images.
* `m.queryIdx` provides the index of the point in the first image (source).
* `m.trainIdx` provides the index of the matched point in the second image (target).
* `.pt` extracts the actual `(x, y)` pixel coordinates.

Next, we format these coordinate lists for OpenCV into a specific NumPy array layout:

```python
import numpy as np

def matched_points(kp1, kp2, matches):
    src = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    dst = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)
    return src, dst

```

* We cast lists to `np.float32` because OpenCV requires 32-bit floating-point coordinates.
* `.reshape(-1, 1, 2)` structures the data into a 3D array of shape $(N, 1, 2)$ where each point has 2 values ($x$ and $y$).

---

## Understanding RANSAC

Raw matches often contain outliers that would pull the estimated transformation off-center, leading to distorted or blurred panoramas.

To solve this, we use **RANSAC (RANdom Sample Consensus)**. It is an iterative estimation algorithm operating in four steps:

1. **Select:** Randomly pick the minimum number of point pairs required to solve a homography ($4$ pairs).
2. **Estimate:** Calculate a candidate homography matrix using only those 4 points.
3. **Test:** Apply this matrix across all remaining matches and evaluate how many align within an error margin (the distance threshold).
4. **Consensus:** Matches that fit within the threshold are labeled **inliers**. The algorithm repeats this process over multiple iterations and keeps the homography model with the highest inlier count.

By leveraging the consensus of the majority, RANSAC ignores noise from dynamic elements, repetitive textures, or mismatches.

---

## Estimating Homography Using RANSAC

A minimum of **4 point pairs** is required to solve the 8 degrees of freedom in a planar homography.

```python
import cv2
import numpy as np

def estimate_homography(kp1, kp2, matches, ransac_threshold=5.0):
    if len(matches) < 4:
        raise ValueError("At least four matches are required")

    src, dst = matched_points(kp1, kp2, matches)
    homography, mask = cv2.findHomography(src, dst, cv2.RANSAC, ransac_threshold)

    if homography is None or mask is None:
        raise ValueError("Homography estimation failed")

    return homography, mask.ravel().astype(bool)

```

### Parameter Breakdown

* `cv2.findHomography(src, dst, cv2.RANSAC, ransac_threshold)`: Computes the $3 \times 3$ transformation matrix using RANSAC.
* `ransac_threshold`: Reprojection error tolerance in pixels (e.g., `5.0` allows a point to land within 5 pixels of its counterpart to count as an inlier).
* `mask.ravel().astype(bool)`: Converts the output mask into a 1D boolean array where `True` flags inliers and `False` flags outliers.

---

## Putting the Pipeline Together

Here is how feature extraction, matching, and robust homography estimation tie together:

```python
# Assume left_image and right_image are already loaded
kp1, des1 = detect_and_compute(left_image)
kp2, des2 = detect_and_compute(right_image)
matches = match_descriptors(des1, des2)

homography, inliers = estimate_homography(
    kp1,
    kp2,
    matches,
    ransac_threshold=5.0,
)

print("matches:", len(matches))
print("inliers:", int(inliers.sum()))
print("inlier ratio:", float(inliers.mean()))
print("homography:")
print(homography)

```

### Example Terminal Output

```text
matches: 120
inliers: 95
inlier ratio: 0.7916666666666666
homography:
[[ 1.250e+00 -1.500e-01  3.000e+02]
 [ 5.000e-02  1.100e+00 -5.000e+01]
 [ 1.000e-04  2.000e-04  1.000e+00]]

```

In this run, out of 120 raw matches, RANSAC identified 95 reliable inliers ($\approx 79\%$ inlier ratio) and produced the final $3 \times 3$ matrix used to warp images into alignment.

---

## Summary and Upcoming Practice

* **Homography:** $3 \times 3$ matrix mapping 2D planar points between scenes (valid for flat surfaces or pure camera rotation).
* **Point Extraction:** Unpacks `.pt` from match indices and formats them into `(N, 1, 2)` `np.float32` arrays.
* **RANSAC:** Robust iterative model fitting using random 4-point samples to discard outlier matches.
* **Threshold & Inliers:** Distinguishes valid geometric correspondence from matching errors via reprojection distance.

## Preparing Points for Homography Estimation

Now that the lesson has shown how a homography links matching points between two images, it is time to put that idea into code by preparing the point arrays that OpenCV needs.

Open geometry.py and complete the two functions:

    In matched_points, build a src array from kp1[m.queryIdx].pt and a dst array from kp2[m.trainIdx].pt. Both should use np.float32, be reshaped to (-1, 1, 2), and then be returned as a tuple (src, dst).
    In estimate_homography, guard against bad input: if there are fewer than four matches, raise a ValueError with the exact message "At least four matches are required".

Getting these helpers right sets the foundation for the RANSAC step you will add next.

```
import numpy as np


def matched_points(kp1, kp2, matches):
    # TODO: Build the `src` array from `kp1[m.queryIdx].pt` using `np.float32`
    # and reshape it to (-1, 1, 2).

    # TODO: Build the `dst` array from `kp2[m.trainIdx].pt` using `np.float32`
    # and reshape it to (-1, 1, 2).

    # TODO: Return both arrays as a tuple (src, dst).
    pass


def estimate_homography(kp1, kp2, matches, ransac_threshold=5.0):
    # TODO: If there are fewer than 4 matches, raise a ValueError with the
    # message "At least four matches are required".
    pass

```

Here is the completed implementation for `geometry.py`:

```python
import numpy as np


def matched_points(kp1, kp2, matches):
    # Build the `src` array from `kp1[m.queryIdx].pt` using `np.float32` and reshape to (-1, 1, 2)
    src = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)

    # Build the `dst` array from `kp2[m.trainIdx].pt` using `np.float32` and reshape to (-1, 1, 2)
    dst = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)

    # Return both arrays as a tuple (src, dst)
    return src, dst


def estimate_homography(kp1, kp2, matches, ransac_threshold=5.0):
    # Guard against fewer than 4 matches
    if len(matches) < 4:
        raise ValueError("At least four matches are required")

```

## Completing the Homography Estimator

With the point preparation and the four-match guard in place, it is time to run the actual RANSAC step and finish the estimate_homography function.

Open geometry.py and follow the TODOs inside estimate_homography:

    Call matched_points with kp1, kp2, and matches, and unpack the result into src and dst.
    Call cv2.findHomography with src, dst, cv2.RANSAC, and ransac_threshold, capturing both the homography matrix and the mask it returns.
    If either the homography or the mask is None, raise a ValueError with the exact message "Homography estimation failed".
    Return the homography along with the mask flattened into a 1D boolean array using mask.ravel().astype(bool).

Once this is done, your function will turn noisy matches into a clean alignment between two images — the core trick that makes panorama stitching possible.

```
import cv2
import numpy as np


def matched_points(kp1, kp2, matches):
    src = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    dst = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)
    return src, dst


def estimate_homography(kp1, kp2, matches, ransac_threshold=5.0):
    if len(matches) < 4:
        raise ValueError("At least four matches are required")

    # TODO: Call `matched_points` with kp1, kp2, and matches, and unpack
    # the result into `src` and `dst`.

    # TODO: Call `cv2.findHomography` with `src`, `dst`, `cv2.RANSAC`, and
    # `ransac_threshold`. Capture both the homography matrix and the mask.

    # TODO: If either the homography or the mask is None, raise a
    # ValueError with the exact message "Homography estimation failed".

    # TODO: Return the homography and the mask flattened into a 1D boolean
    # array using `mask.ravel().astype(bool)`.

```

Here is the completed implementation for `geometry.py`:

```python
import cv2
import numpy as np


def matched_points(kp1, kp2, matches):
    src = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    dst = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)
    return src, dst


def estimate_homography(kp1, kp2, matches, ransac_threshold=5.0):
    if len(matches) < 4:
        raise ValueError("At least four matches are required")

    # 1. Unpack matched point coordinates into src and dst
    src, dst = matched_points(kp1, kp2, matches)

    # 2. Compute homography matrix and inlier mask via RANSAC
    homography, mask = cv2.findHomography(src, dst, cv2.RANSAC, ransac_threshold)

    # 3. Check for estimation failure
    if homography is None or mask is None:
        raise ValueError("Homography estimation failed")

    # 4. Return homography matrix and flattened 1D boolean inlier mask
    return homography, mask.ravel().astype(bool)

```

## Wiring Up the Command Line

Nice work wrapping up geometry.py with a robust homography estimator! Now, it is time to start building the command-line driver that will actually use it on a pair of images.

In this exercise, you will set up the argument parser inside main() so the script can be called like python solution.py left.jpg right.jpg --method sift.

Open solution.py and follow the TODO comments to:

    Create an ArgumentParser.
    Add two positional arguments named left and right.
    Add --method with choices=["sift", "orb", "akaze"] and default="sift".
    Add --ratio with type=float and default=0.75.
    Add --ransac-threshold with type=float and default=5.0.
    Parse the arguments into a variable called args.

The rest of main() will be filled in during the next exercise, so for now, just focus on getting the CLI right. A solid argument parser makes the whole pipeline easy to test and reuse!

```
import argparse

from cvkit import preprocess_for_features, read_color
from features import detect_and_compute, match_descriptors
from geometry import estimate_homography


def main():
    # TODO: Create an ArgumentParser.

    # TODO: Add a positional argument named "left".

    # TODO: Add a positional argument named "right".

    # TODO: Add an optional argument "--method" with
    # choices=["sift", "orb", "akaze"] and default="sift".

    # TODO: Add an optional argument "--ratio" with type=float and
    # default=0.75.

    # TODO: Add an optional argument "--ransac-threshold" with type=float
    # and default=5.0.

    # TODO: Parse the arguments and store the result in `args`.
    pass

    # The rest of main() will be implemented in the next exercise.


if __name__ == "__main__":
    main()

```

Here is the completed `solution.py` with the argument parser configured according to the requirements:

```python
import argparse

from cvkit import preprocess_for_features, read_color
from features import detect_and_compute, match_descriptors
from geometry import estimate_homography


def main():
    # 1. Create an ArgumentParser
    parser = argparse.ArgumentParser()

    # 2. Positional arguments for image paths
    parser.add_argument("left", help="Path to the left image")
    parser.add_argument("right", help="Path to the right image")

    # 3. Optional arguments with specified choices and defaults
    parser.add_argument(
        "--method",
        choices=["sift", "orb", "akaze"],
        default="sift",
        help="Feature detection method (default: sift)",
    )
    parser.add_argument(
        "--ratio",
        type=float,
        default=0.75,
        help="Lowe's ratio test threshold (default: 0.75)",
    )
    parser.add_argument(
        "--ransac-threshold",
        type=float,
        default=5.0,
        help="RANSAC reprojection error threshold (default: 5.0)",
    )

    # 4. Parse the arguments into `args`
    args = parser.parse_args()

    # The rest of main() will be implemented in the next exercise.


if __name__ == "__main__":
    main()

```

## Running the Full Alignment Pipeline